# Traceprop-LLM — inline (trajectory) vs. final-checkpoint gradients

**Question (MLSys reviewer, item 9):** exp27/exp28's LDS numbers are computed by calling `LoRAGradientLogger` *after* training finishes, on the frozen final model — a single post-hoc forward+backward pass. That's architecturally identical to what TRAK/LoGRA do (recompute per-sample gradients at one checkpoint). It is **not** what the overhead experiments (exp25/exp26) measure, which capture gradients live, at whatever the parameters were at each training step.

This notebook trains **one** target model once. During that single run it accumulates the inline (trajectory-sum) per-example train gradient — `g_inline(i) = sum_t g(i; theta_t)`, TracIn-style — alongside the existing post-hoc final-checkpoint gradient (same `LoRAGradientLogger`, same tracked scope). Test gradients are evaluated once, at the final model, for both conditions. Both train-side quantities are then scored against the *same* subset-retraining ground truth, so the comparison isolates exactly the one variable the reviewer asked about.

**CPU proxy result already run (tiny synthetic backend, 500 subsets):**

| | dot-product | TRAK |
|---|---|---|
| final-checkpoint | 0.6511 ± 0.0639 | 0.3654 ± 0.0598 |
| inline (trajectory-sum) | **0.7017 ± 0.0644** | 0.1080 ± 0.0429 |
| random | 0.0078 ± 0.0595 | — |

Dot-product: inline matches or beats final-checkpoint. TRAK: inline is much worse (its inverse-Gram correction assumes gradients come from one fixed linearization point, which the trajectory sum violates). This cell block runs the same ablation on the real paper setup: GPT-2 + LoRA on SST-2.

In [ ]:
!pip -q install transformers peft datasets accelerate
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .

## Inline vs. final-checkpoint LDS: GPT-2 + LoRA on SST-2

Start with 64 subsets (~15-25 min, since this trains once with inline capture, then does a post-hoc pass, then N subset retrains). Bump `--n_subsets` to 500 for the number that goes in the paper (`\S\ref{sec:lds}` currently reports SST-2 results at 500 subsets — match that).

In [ ]:
%cd /content/Traceprop/experiments
!python exp29_inline_vs_final_lds.py --backend hf --model gpt2 --device cuda \
    --data sst2 --n_train 1000 --n_test 200 --n_subsets 64 --epochs 3 --seq 64 \
    --batch 16 --proj_dim 256 --track 1

## 500-subset run (the paper number)

Only run this once the 64-subset smoke test above looks sane (nonzero accuracy, LDS values roughly in the same ballpark as exp27's SST-2 numbers). This will take much longer — budget accordingly.

In [ ]:
%cd /content/Traceprop/experiments
!python exp29_inline_vs_final_lds.py --backend hf --model gpt2 --device cuda \
    --data sst2 --n_train 1000 --n_test 200 --n_subsets 500 --epochs 3 --seq 64 \
    --batch 16 --proj_dim 256 --track 1

## Result

Look for `inline_dot` vs `final_dot`, and `inline_trak` vs `final_trak`. If the CPU-proxy pattern holds (dot: inline ≥ final; TRAK: inline << final), that's the real-setting confirmation for item 9 — write it up as: use the dot-product estimator with inline capture (it's not a downgrade, and the 117–154× speedup is a genuine apples-to-apples win), and flag TRAK as incompatible with trajectory-summed gradients specifically.

In [ ]:
import json, glob
for p in glob.glob('/content/Traceprop/experiments/results/exp29_hf_*.json'):
    d = json.load(open(p))
    print(d['model'], 'acc', d['target_test_acc'], 'subsets', d['n_subsets'])
    for k, v in d['lds'].items():
        print(f"  {k:<14} {v['mean']:+.4f} ± {v['std']:.4f}")